# Partie 1 : ABA generator
Le moteur est dans `aba.py` ; ce notebook démontre chaque capacité demandée.

## 1. Parsing de l'exemple du sujet

In [1]:
from aba import *
fw = ABAFramework.from_text(EXAMPLE)
print(fw.to_text())
print('plat:', fw.is_flat(), '| circulaire:', fw.is_circular(), '| atomique:', fw.is_atomic())

## Literals
L: [a,b,c,q,p,r,s,t]

## Assumptions
A: [a,b,c]

## Contraries
C(a): r
C(b): s
C(c): t

## Rules
[r1]: p <- q,a
[r2]: q <-
[r3]: r <- b,c
[r4]: t <- p,c
[r5]: s <- t

## Preferences
PREF: a > b

plat: True | circulaire: False | atomique: False


## 2. Conversion atomique sensitive

In [2]:
at = fw.prepare()
print(at.to_text())
print('atomique:', at.is_atomic())

## Literals
L: [a,b,c,q,p,r,s,t,q_d,q_nd,p_d,p_nd,r_d,r_nd,s_d,s_nd,t_d,t_nd]

## Assumptions
A: [a,b,c,q_d,q_nd,p_d,p_nd,r_d,r_nd,s_d,s_nd,t_d,t_nd]

## Contraries
C(a): r
C(b): s
C(c): t
C(q_d): q_nd
C(q_nd): q
C(p_d): p_nd
C(p_nd): p
C(r_d): r_nd
C(r_nd): r
C(s_d): s_nd
C(s_nd): s
C(t_d): t_nd
C(t_nd): t

## Rules
[r1]: p <- q_d,a
[r2]: q <-
[r3]: r <- b,c
[r4]: t <- p_d,c
[r5]: s <- t_d

## Preferences
PREF: a > b

atomique: True


## 3. Arguments (avec arbres de dérivation) et attaques sans préférences

In [3]:
args = fw.arguments()
for a in args: print(a)
print()
print(args[7].tree_str())
print()
for t in fw.attacks(args, use_preferences=False): print(t)

A1: {} ⊢ q
A2: {a} ⊢ a
A3: {b} ⊢ b
A4: {c} ⊢ c
A5: {a} ⊢ p
A6: {b,c} ⊢ r
A7: {a,c} ⊢ s
A8: {a,c} ⊢ t

t   [r4]
  p   [r1]
    q   [r2]
    a   (hypothèse)
  c   (hypothèse)

A6 -> A2  (normal, via a)
A6 -> A5  (normal, via a)
A6 -> A7  (normal, via a)
A6 -> A8  (normal, via a)
A7 -> A3  (normal, via b)
A7 -> A6  (normal, via b)
A8 -> A4  (normal, via c)
A8 -> A6  (normal, via c)
A8 -> A7  (normal, via c)
A8 -> A8  (normal, via c)


## 4. Préférences `a > b` : attaques normales / inversées

In [4]:
for t in fw.attacks(args): print(t)

A2 -> A6  (reverse, via b)
A5 -> A6  (reverse, via b)
A7 -> A3  (normal, via b)
A7 -> A6  (normal, via b)
A7 -> A6  (reverse, via b)
A8 -> A4  (normal, via c)
A8 -> A6  (normal, via c)
A8 -> A6  (reverse, via b)
A8 -> A7  (normal, via c)
A8 -> A8  (normal, via c)


In [5]:
from collections import Counter
print(Counter(t.kind for t in fw.attacks(args)))

Counter({'normal': 6, 'reverse': 4})


## 5. Cas supplémentaires
### 5a. Règles circulaires

In [6]:
circ = ABAFramework.from_text('''
L: [a,b,p,q,r]
A: [a,b]
C(a): q
C(b): r
[r1]: p <- q,a
[r2]: q <- p
[r3]: q <- b
''')
print('cycle:', circ.find_cycle())
nc = circ.to_non_circular(); print('non circulaire après conversion:', not nc.is_circular())
print(nc.to_atomic().to_text())
for a in circ.arguments(): print(a)

cycle: ['p', 'q', 'p']
non circulaire après conversion: True
## Literals
L: [a,b,p,q,r,p_1,p_2,q_1,q_2,r_1,r_2,p_d,p_nd,q_d,q_nd,r_d,r_nd,p_1_d,p_1_nd,p_2_d,p_2_nd,q_1_d,q_1_nd,q_2_d,q_2_nd,r_1_d,r_1_nd,r_2_d,r_2_nd]

## Assumptions
A: [a,b,p_d,p_nd,q_d,q_nd,r_d,r_nd,p_1_d,p_1_nd,p_2_d,p_2_nd,q_1_d,q_1_nd,q_2_d,q_2_nd,r_1_d,r_1_nd,r_2_d,r_2_nd]

## Contraries
C(a): q
C(b): r
C(p_d): p_nd
C(p_nd): p
C(q_d): q_nd
C(q_nd): q
C(r_d): r_nd
C(r_nd): r
C(p_1_d): p_1_nd
C(p_1_nd): p_1
C(p_2_d): p_2_nd
C(p_2_nd): p_2
C(q_1_d): q_1_nd
C(q_1_nd): q_1
C(q_2_d): q_2_nd
C(q_2_nd): q_2
C(r_1_d): r_1_nd
C(r_1_nd): r_1
C(r_2_d): r_2_nd
C(r_2_nd): r_2

## Rules
[r1.2]: p_2 <- q_1_d,a
[r1.3]: p <- q_2_d,a
[r2.2]: q_2 <- p_1_d
[r2.3]: q <- p_2_d
[r3.1]: q_1 <- b
[r3.2]: q_2 <- b
[r3.3]: q <- b

A1: {a} ⊢ a
A2: {b} ⊢ b
A3: {b} ⊢ q
A4: {a,b} ⊢ p
A5: {a,b} ⊢ q


### 5b. Hypothèse en tête de règle (conversion en ABA plat)

In [7]:
nf = ABAFramework.from_text('''
L: [a,b,c,x]
A: [a,b]
C(a): c
C(b): x
[r1]: a <- b
[r2]: c <-
''')
print('plat:', nf.is_flat()); print(nf.to_flat().to_text())

plat: False
## Literals
L: [a,b,c,x,a']

## Assumptions
A: [a',b]

## Contraries
C(a'): c
C(b): x

## Rules
[r1]: a <- b
[r2]: c <-
[f_a]: a <- a'



### 5c. Préférences d'équivalence `a,b > c` et chaînes `a > b > c`

In [8]:
p = ABAFramework.from_text('''
L: [a,b,c,x,y,z]
A: [a,b,c]
C(a): x
C(b): y
C(c): z
[r1]: x <- b
[r2]: y <- a
[r3]: z <- a
PREF: a,b > c
''')
print('a>c', p.strictly_preferred('a','c'), '| b>c', p.strictly_preferred('b','c'), '| a>b', p.strictly_preferred('a','b'))
A = p.arguments()
for a in A: print(a)
for t in p.attacks(A): print(t)

a>c True | b>c True | a>b False
A1: {a} ⊢ a
A2: {b} ⊢ b
A3: {c} ⊢ c
A4: {b} ⊢ x
A5: {a} ⊢ y
A6: {a} ⊢ z
A4 -> A1  (normal, via a)
A4 -> A5  (normal, via a)
A4 -> A6  (normal, via a)
A5 -> A2  (normal, via b)
A5 -> A4  (normal, via b)
A6 -> A3  (normal, via c)


## 6. Vérification TD4 : conversion non circulaire et framework atomique

In [9]:
td4 = ABAFramework.from_text('''
L: [a,b,x,y,z]
A: [a,b]
C(a): y
C(b): x
[r1]: y <- b
[r2]: y <- y
[r3]: x <- a
[r4]: x <- x
[r5]: z <- x,y
''')
td4_nc = td4.to_non_circular()
print(td4_nc.to_text())
print('arguments non circulaires:', len(td4_nc.arguments()))
td4_atomic = td4_nc.to_atomic()
print('hypothèses atomiques:', len(td4_atomic.assumptions))
print('règles atomiques:', len(td4_atomic.rules))
print('arguments atomiques:', len(td4_atomic.arguments()))
print('vérification du compte atomique:', len(td4_atomic.arguments()) == len(td4_atomic.assumptions) + len(td4_atomic.rules))

## Literals
L: [a,b,x,y,z,x_1,x_2,y_1,y_2,z_1,z_2]

## Assumptions
A: [a,b]

## Contraries
C(a): y
C(b): x

## Rules
[r1.1]: y_1 <- b
[r1.2]: y_2 <- b
[r1.3]: y <- b
[r2.2]: y_2 <- y_1
[r2.3]: y <- y_2
[r3.1]: x_1 <- a
[r3.2]: x_2 <- a
[r3.3]: x <- a
[r4.2]: x_2 <- x_1
[r4.3]: x <- x_2
[r5.2]: z_2 <- x_1,y_1
[r5.3]: z <- x_2,y_2

arguments non circulaires: 10
hypothèses atomiques: 20
règles atomiques: 12
arguments atomiques: 32
vérification du compte atomique: True


## 7. Vérification TD4 : attaques ABA+ entre ensembles

In [10]:
def format_set(values):
	return '∅' if not values else '{' + ','.join(sorted(values)) + '}'

normal, reverse = fw.set_attacks()
print('attaques normales:', len(normal))
print('attaques inversées:', len(reverse))
for X, Y in sorted(normal): print('normal:', format_set(X), '->', format_set(Y))
for X, Y in sorted(reverse): print('reverse:', format_set(X), '->', format_set(Y))

attaques normales: 12
attaques inversées: 8
normal: {a,c} -> {b}
normal: {a,c} -> {a,b}
normal: {a,c} -> {c}
normal: {a,c} -> {a,c}
normal: {a,c} -> {b,c}
normal: {a,c} -> {a,b,c}
normal: {a,b,c} -> {b}
normal: {a,b,c} -> {b,c}
normal: {a,b,c} -> {a,b}
normal: {a,b,c} -> {c}
normal: {a,b,c} -> {a,c}
normal: {a,b,c} -> {a,b,c}
reverse: {a} -> {b,c}
reverse: {a} -> {a,b,c}
reverse: {a,b} -> {b,c}
reverse: {a,c} -> {b,c}
reverse: {a,c} -> {a,b,c}
reverse: {a,b} -> {a,b,c}
reverse: {a,b,c} -> {b,c}
reverse: {a,b,c} -> {a,b,c}


## 8. Graphe (Graphviz)

In [11]:
print(fw.to_dot(args))

digraph ABA {
  rankdir=LR;
  node [shape=box, style=rounded, fontname="Helvetica"];
  A1 [label="A1\n{} ⊢ q"];
  A2 [label="A2\n{a} ⊢ a"];
  A3 [label="A3\n{b} ⊢ b"];
  A4 [label="A4\n{c} ⊢ c"];
  A5 [label="A5\n{a} ⊢ p"];
  A6 [label="A6\n{b,c} ⊢ r"];
  A7 [label="A7\n{a,c} ⊢ s"];
  A8 [label="A8\n{a,c} ⊢ t"];
  A2 -> A6 [color="red", style=dashed];
  A5 -> A6 [color="red", style=dashed];
  A7 -> A3 [color="black"];
  A7 -> A6 [color="black"];
  A7 -> A6 [color="red", style=dashed];
  A8 -> A4 [color="black"];
  A8 -> A6 [color="black"];
  A8 -> A6 [color="red", style=dashed];
  A8 -> A7 [color="black"];
  A8 -> A8 [color="black"];
}
